# TP — Détection d'anomalies par embeddings auto-supervisés (MVTec AD, classe `bottle`)

**Modèles** : DINOv3 ViT-H+/16 et I-JEPA ViT-H/14 (embeddings auto-supervisés)  
**Méthodes** : distance au centroïde des images normales vs K-means (k=2)

Plan : (1) données, (2) extraction des embeddings, (3) baseline centroïde, (4) K-means k=2, (5) bonus, (6) synthèse.

> Règle d'or : le centroïde (ou tout modèle de « normalité ») est estimé **uniquement** sur `train/good`.

In [ ]:
!pip -q install transformers accelerate scikit-learn matplotlib pandas tqdm

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, adjusted_rand_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.neighbors import NearestNeighbors

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_ROOT = Path("./mvtec_ad/bottle")   # a adapter : dossier contenant train/ et test/
BATCH = 16
CACHE = Path("./cache"); CACHE.mkdir(exist_ok=True)
print("device:", DEVICE)

## 1. Données

Structure attendue : `bottle/train/good/*.png` et `bottle/test/{good,broken_large,broken_small,contamination}/*.png`.  
Construisez `train_paths`, `test_paths`, `test_labels` (0 = normal, 1 = anomalie) et `test_types` (nom du sous-dossier).

In [ ]:
def list_images(folder):
    # TODO : retourner la liste triee des fichiers .png d'un dossier
    raise NotImplementedError

# TODO : construire train_paths (train/good)
train_paths = ...

# TODO : parcourir les sous-dossiers de test/ et construire
# test_paths (liste), test_labels (np.array 0/1), test_types (np.array de str)
test_paths, test_labels, test_types = ..., ..., ...

print(len(train_paths), "train/good |", len(test_paths), "test")
print(pd.Series(test_types).value_counts())

In [ ]:
assert len(train_paths) > 0 and len(test_paths) == len(test_labels) == len(test_types)
fig, axs = plt.subplots(1, 4, figsize=(14, 4))
for ax, t in zip(axs, ["good", "broken_large", "broken_small", "contamination"]):
    i = int(np.where(test_types == t)[0][0])
    ax.imshow(Image.open(test_paths[i])); ax.set_title(t); ax.axis("off")
plt.show()

## 2. Extraction des embeddings

- **DINOv3 ViT-H+/16** via `transformers` (miroir non restreint `LibreYOLO/LibreDINOv3vith16plus`, chargé en bfloat16) : embedding = `pooler_output` (token CLS) ou moyenne des tokens de patchs.
- **I-JEPA ViT-H/14** via `transformers` (`facebook/ijepa_vith14_1k`) : pas de token CLS, on moyenne les tokens de `last_hidden_state`.

Le prétraitement (redimensionnement, normalisation) est géré par le processeur de chaque modèle. Les embeddings sont L2-normalisés plus tard (section 3), pas dans l'extraction, pour pouvoir les mettre en cache bruts.

Si la mémoire GPU est limitée, réduisez `BATCH` ou traitez un seul modèle (retirez l'autre de `MODELS`).

In [ ]:
from transformers import AutoImageProcessor, AutoModel

def load_hf(model_id, dtype=torch.float32, **kw):
    processor = AutoImageProcessor.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id, dtype=dtype, device_map="auto", **kw).eval()
    return processor, model

def load_dinov3():
    return load_hf("LibreYOLO/LibreDINOv3vith16plus", dtype=torch.bfloat16)

def load_ijepa():
    return load_hf("facebook/ijepa_vith16_1k", attn_implementation="sdpa")

def _prepare_inputs(processor, model, images):
    inputs = processor(images=images, return_tensors="pt")
    # 1) device pour tout
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    # 2) dtype uniquement pour les tenseurs flottants
    inputs = {k: v.to(model.dtype) if v.is_floating_point() else v
              for k, v in inputs.items()}
    return inputs


def embed_dinov3(bundle, images, pool="cls"):
    """
    Extrait un embedding global par image avec DINOv3.

    Paramètres
    ----------
    bundle : tuple (processor, model)
        Processeur et modèle Hugging Face déjà chargés.
    images : PIL.Image ou liste de PIL.Image
        Image(s) à encoder.
    pool : {"cls", "mean"}
        - "cls"  : utilise le token CLS (sortie du pooler ou last_hidden_state[:, 0]).
        - "mean" : moyenne des tokens de patch, en excluant CLS et les registres.

    Retour
    ------
    torch.Tensor de forme (B, D)
    """
    processor, model = bundle

    # Prépare les entrées : device pour tout, dtype modèle uniquement pour le flottant.
    # (évite de caster pixel_mask / image_sizes en bfloat16, ce qui casserait le modèle)
    inputs = _prepare_inputs(processor, model, images)

    # Forward pass (sans gradient en amont si le modèle est gelé)
    out = model(**inputs)

    if pool == "cls":
        # DINOv3 expose un pooler_output (CLS après tête de pooling).
        # Si le modèle est chargé via AutoModel sans tête de pooling,
        # pooler_output peut être None : dans ce cas, utiliser last_hidden_state[:, 0].
        return out.pooler_output

    # --- Pooling moyen sur les tokens de patch uniquement ---
    #
    # La séquence de sortie de DINOv3 est structurée ainsi :
    #   [ CLS ] [ registre_1 ] ... [ registre_R ] [ patch_1 ] ... [ patch_N ]
    #
    # - 1 token CLS (indice 0)
    # - R tokens de registre (R = model.config.num_register_tokens, souvent 0 ou 4)
    # - N tokens de patch (la partie qui nous intéresse pour un embedding "image")
    #
    # On doit donc sauter les 1 + R premiers tokens pour ne moyenner QUE les patchs.
    # Si on oubliait ce skip, la moyenne serait contaminée par CLS et les registres,
    # qui ont des rôles différents (résumé global / artefacts de calcul).
    n_skip = 1 + getattr(model.config, "num_register_tokens", 0)  # CLS + registres

    # last_hidden_state : (B, 1 + R + N, D)
    # [:, n_skip:]      : (B, N, D)  -> on garde uniquement les patchs
    # .mean(dim=1)      : (B, D)     -> moyenne sur les patchs
    return out.last_hidden_state[:, n_skip:].mean(dim=1)


def embed_ijepa(bundle, images):
    processor, model = bundle
    inputs = _prepare_inputs(processor, model, images)
    return model(**inputs).last_hidden_state.mean(dim=1)

@torch.inference_mode()
def extract(paths, bundle, embed_fn, cache_name):
    # TODO : boucle par batchs de taille BATCH
    #   - ouvrir les images (PIL, RGB), appeler embed_fn(bundle, imgs)
    #   - passer en float32 numpy (.float().cpu().numpy()), concatener
    #   - sauvegarder dans CACHE/f"{cache_name}.npy" (recharger si le fichier existe)
    raise NotImplementedError

In [ ]:
MODELS = {
    "DINOv3-H+/16": (load_dinov3, embed_dinov3),
    "I-JEPA-H/16": (load_ijepa, embed_ijepa),
}

feats = {}
for name, (loader, fn) in MODELS.items():
    tag = name.replace("/", "").replace("-", "_").replace("+", "p")
    bundle = loader()
    feats[name] = {
        "train": extract(train_paths, bundle, fn, f"{tag}_train"),
        "test":  extract(test_paths,  bundle, fn, f"{tag}_test"),
    }
    del bundle
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(name, feats[name]["train"].shape, feats[name]["test"].shape)
    assert feats[name]["train"].shape[0] == len(train_paths)
    assert feats[name]["test"].shape[0] == len(test_paths)

## 3. Baseline : distance au centroïde (Partie A)

1. Normalisez les embeddings en L2.
2. Centroïde = moyenne des embeddings de `train/good`.
3. Score d'anomalie = distance L2 au centroïde.
4. AUROC image-level (global, puis par type de défaut — attention, seulement 20 images `good` : estimations bruitées).

In [ ]:
def centroid_scores(z_train, z_test):
    # TODO : normalisation L2, centroide de z_train, distance L2 de chaque z_test
    raise NotImplementedError

def evaluate(scores, labels=None, types=None):
    # TODO : retourner un dict {"F1": x}
    raise NotImplementedError

results = []
scores_centroid = {}
for name in feats:
    # TODO : calculer les scores, les stocker, evaluer, ajouter une ligne a results
    pass
pd.DataFrame(results).round(3)

In [ ]:
fig, axs = plt.subplots(1, len(feats), figsize=(11, 3.5))
for ax, name in zip(np.atleast_1d(axs), feats):
    s = scores_centroid[name]
    ax.hist(s[test_labels == 0], bins=15, alpha=.6, label="good")
    ax.hist(s[test_labels == 1], bins=15, alpha=.6, label="anomalie")
    ax.set_title(name); ax.set_xlabel("distance au centroide"); ax.legend()
plt.tight_layout(); plt.show()

## 4. K-means k=2 (Partie B, exploratoire)

On clusterise `train/good` **et** `test` ensemble, ce qui constitue une fuite
de données dans un vrai scénario industriel — à documenter comme limite
méthodologique.

1. Appliquer `KMeans(2)` (scikit-learn, `random_state` fixé) sur les embeddings
   L2-normalisés concaténés `[z_train ; z_test]`.
2. Comparer les clusters aux vrais labels de test via `pd.crosstab` :

   | Vrai label \ Cluster | Cluster 0 | Cluster 1 | Total |
   |:---------------------|----------:|----------:|------:|
   | `defect`             |        40 |        23 |    63 |
   | `good`               |        15 |         5 |    20 |

   **Lecture** : les deux clusters contiennent une majorité de `defect`
   (40/63 et 23/63). K-means n'a donc pas séparé normal vs anomalie ; il a
   capté un autre facteur de variance (éclairage, pose, etc.).

3. Définir le cluster « normal » comme celui qui contient le plus d'images
   de `train/good`.

4. Visualiser en PCA 2D :
   - figure 1 : couleur = cluster K-means ;
   - figure 2 : couleur = type réel (`good` / `defect`).



In [ ]:
km_store = {}
for name in feats:
    ztr, zte = normalize(feats[name]["train"]), normalize(feats[name]["test"])
    n_tr = len(ztr)
    # TODO 1 : Z = concatenation train+test ; KMeans(2, n_init=10, random_state=SEED)
    # TODO 2 : crosstab(test_types, clusters_test) 
    # TODO 3 : cluster "normal" = cluster majoritaire parmi les images de train
    # TODO 4 : trois scores (hard / distance au cluster normal / distance a son cluster)
    #          -> F1, ajoute a results avec method = "kmeans_..."
    # TODO 5 : km_store[name] = (Z, labels, n_tr)
    pass

pd.DataFrame(results).round(3)

In [ ]:
# TODO : figure a 2 colonnes par modele : PCA 2D coloree par cluster, puis par type reel
# (PCA 2D = visualisation uniquement, jamais pour la detection)

**Questions de discussion** (répondre par écrit, 5 à 10 lignes au total)

1. Les deux clusters de K-means k=2 correspondent-ils à « normal / anomalie » ? Qu'indiquent le `crosstab` ? Quel critère de variance le clustering a-t-il probablement capté ?
2. Pourquoi K-means k=2 n'est-il pas un détecteur d'anomalies (notion de normalité, dépendance à la proportion d'anomalies dans le test, absence de score natif) ?
3. Quelle fuite méthodologique y a-t-il à clusteriser train+test ? Comment les variantes « train-only » du bonus la corrigent-elles ?
4. DINOv3-H+ vs I-JEPA-H (deux très gros modèles) : l'un domine-t-il l'autre ici ? Le coût mémoire/temps est-il justifié pour une classe aussi simple que `bottle` ? Quel effet a le choix du pooling (CLS vs moyenne) ?
5. Pourquoi les AUROC par type de défaut sont-ils à interpréter avec prudence ?